## 0. Install dependencies

In [12]:
%pip install -q pandas numpy matplotlib seaborn plotly scipy scikit-learn openpyxl ipywidgets

Note: you may need to restart the kernel to use updated packages.


## 1. Imports and settings

In [13]:
from pathlib import Path
import json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

OUTPUT_DIR = Path("vss_ai_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42
print("VSS AI notebook ready. Outputs:", OUTPUT_DIR.resolve())

VSS AI notebook ready. Outputs: D:\Programing\Programing_folder\Program Files\projectfile\vayapar_sangisarathi\notebooks\vss_ai_outputs


## 2. Select a CSV or Excel file


In [14]:
FILE_PATH = r"D:\Programing\Programing_folder\Program Files\projectfile\vayapar_sangisarathi\notebooks\sales_data_sample.csv"  # Enter your file path here

# Optional upload widget for Jupyter
try:
    import ipywidgets as widgets
    uploader = widgets.FileUpload(accept=".csv,.xlsx,.xls", multiple=False)
    display(uploader)
except Exception as e:
    uploader = None
    print("Upload widget unavailable; use FILE_PATH. Reason:", e)

def save_uploaded_file(widget):
    if widget is None or not widget.value:
        return None
    value = widget.value
    if isinstance(value, tuple):  # ipywidgets 8
        item = value[0]
        name, content = item["name"], item["content"]
    else:  # ipywidgets 7
        name, item = next(iter(value.items()))
        content = item["content"]
    suffix = Path(name).suffix.lower()
    if suffix not in {".csv", ".xlsx", ".xls"}:
        raise ValueError("Supported formats: CSV, XLSX, XLS")
    path = Path("uploaded_" + Path(name).name)
    path.write_bytes(bytes(content))
    return path

uploaded_path = save_uploaded_file(uploader) if "uploader" in globals() else None
selected_path = Path(FILE_PATH) if FILE_PATH else uploaded_path
print("Selected file:", selected_path)

FileUpload(value=(), accept='.csv,.xlsx,.xls', description='Upload')

Selected file: D:\Programing\Programing_folder\Program Files\projectfile\vayapar_sangisarathi\notebooks\sales_data_sample.csv


## 3. Load dataset

In [15]:
def load_dataset(path, sheet_name=0):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")
    suffix = path.suffix.lower()
    if suffix == ".csv":
        for encoding in ["utf-8-sig", "utf-8", "cp1252", "latin1"]:
            try:
                frame = pd.read_csv(path, encoding=encoding, low_memory=False)
                print("CSV encoding:", encoding)
                return frame
            except UnicodeDecodeError:
                continue
        raise ValueError("Could not decode CSV. Check its encoding.")
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, sheet_name=sheet_name)
    raise ValueError("Upload a CSV, XLSX or XLS file.")

if selected_path is None:
    raise ValueError("Set FILE_PATH or upload a file, then rerun this cell.")
df_raw = load_dataset(selected_path)
df = df_raw.copy(deep=True)
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
display(df.head())

CSV encoding: cp1252
Loaded 2,823 rows × 25 columns


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,PRODUCTLINE,MSRP,PRODUCTCODE,CUSTOMERNAME,PHONE,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,Motorcycles,95,S10_1678,Land of Toys Inc.,2125557818,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,Motorcycles,95,S10_1678,Reims Collectables,26.47.1555,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,Motorcycles,95,S10_1678,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,Motorcycles,95,S10_1678,Toys4GrownUps.com,6265557265,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,Motorcycles,95,S10_1678,Corporate Gift Ideas Co.,6505551386,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


## 4. Dataset overview

In [16]:
overview = pd.DataFrame({
    "metric": ["Rows", "Columns", "Exact duplicate rows", "Memory MB"],
    "value": [
        len(df), df.shape[1], int(df.duplicated().sum()),
        round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    ]
})
display(overview)
print("Columns:", df.columns.tolist())
df.info()

,metric,value
0,Rows,2823.00
1,Columns,25.00
2,Exact duplicate rows,0.00
3,Memory MB,2.89


Columns: ['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID', 'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'DEALSIZE']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2823 entries, 0 to 2822
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDERNUMBER       2823 non-null   int64  
 1   QUANTITYORDERED   2823 non-null   int64  
 2   PRICEEACH         2823 non-null   float64
 3   ORDERLINENUMBER   2823 non-null   int64  
 4   SALES             2823 non-null   float64
 5   ORDERDATE         2823 non-null   object 
 6   STATUS            2823 non-null   object 
 7   QTR_ID            2823 non-null   int64  
 8   MONTH_ID          2823 non-null   int64  
 9   YEAR_ID           2823 non-nu

In [17]:
def infer_type(series, name):
    s = series.dropna()
    n = str(name).lower()
    if s.empty: return "empty"
    if pd.api.types.is_bool_dtype(series): return "boolean"
    if pd.api.types.is_datetime64_any_dtype(series): return "datetime"
    if pd.api.types.is_numeric_dtype(series):
        return "numeric_discrete" if s.nunique() <= 20 else "numeric_continuous"
    if any(k in n for k in ["date", "time", "timestamp"]):
        parsed = pd.to_datetime(s.astype(str), errors="coerce")
        if parsed.notna().mean() >= .8: return "datetime_candidate"
    ratio = s.nunique() / max(len(s), 1)
    if s.nunique() == 2: return "binary_categorical"
    if ratio <= .05 or s.nunique() <= 30: return "categorical"
    return "text_or_identifier"

profile_rows = []
for col in df.columns:
    s = df[col]
    profile_rows.append({
        "column": col, "pandas_dtype": str(s.dtype),
        "inferred_type": infer_type(s, col),
        "non_null": int(s.notna().sum()),
        "missing": int(s.isna().sum()),
        "missing_pct": round(s.isna().mean()*100, 2),
        "unique": int(s.nunique(dropna=True)),
        "sample_values": ", ".join(map(str, s.dropna().head(3).tolist()))
    })
column_profile = pd.DataFrame(profile_rows)
display(column_profile)

,column,pandas_dtype,inferred_type,non_null,missing,missing_pct,unique,sample_values
0,ORDERNUMBER,int64,numeric_continuous,2823,0,0.00,307,"10107, 10121, 10134"
1,QUANTITYORDERED,int64,numeric_continuous,2823,0,0.00,58,"30, 34, 41"
2,PRICEEACH,float64,numeric_continuous,2823,0,0.00,1016,"95.7, 81.35, 94.74"
3,ORDERLINENUMBER,int64,numeric_discrete,2823,0,0.00,18,"2, 5, 2"
4,SALES,float64,numeric_continuous,2823,0,0.00,2763,"2871.0, 2765.9, 3884.34"
5,ORDERDATE,object,datetime_candidate,2823,0,0.00,252,"2/24/2003 0:00, 5/7/2003 0:00, 7/1/2003 0:00"
6,STATUS,object,categorical,2823,0,0.00,6,"Shipped, Shipped, Shipped"
7,QTR_ID,int64,numeric_discrete,2823,0,0.00,4,"1, 2, 3"
8,MONTH_ID,int64,numeric_discrete,2823,0,0.00,12,"2, 5, 7"
9,YEAR_ID,int64,numeric_discrete,2823,0,0.00,3,"2003, 2003, 2003"


## 6. Data-quality report

In [18]:
quality_report = pd.DataFrame({
    "column": df.columns,
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_pct": [round(df[c].isna().mean()*100, 2) for c in df.columns],
    "unique_count": [int(df[c].nunique(dropna=True)) for c in df.columns],
    "constant_column": [df[c].nunique(dropna=True) <= 1 for c in df.columns]
}).sort_values("missing_pct", ascending=False)
display(quality_report)
print("Exact duplicate rows:", int(df.duplicated().sum()))

,column,missing_count,missing_pct,unique_count,constant_column
16,ADDRESSLINE2,2521,89.30,9,False
18,STATE,1486,52.64,16,False
21,TERRITORY,1074,38.04,3,False
19,POSTALCODE,76,2.69,73,False
0,ORDERNUMBER,0,0.00,307,False
13,CUSTOMERNAME,0,0.00,92,False
23,CONTACTFIRSTNAME,0,0.00,72,False
22,CONTACTLASTNAME,0,0.00,77,False
20,COUNTRY,0,0.00,19,False
17,CITY,0,0.00,73,False


Exact duplicate rows: 0


## 7. Missing-value analysis

In [19]:
missing = df.isna().sum().rename("missing_count").to_frame()
missing["missing_pct"] = (missing["missing_count"] / max(len(df), 1) * 100).round(2)
display(missing.sort_values("missing_count", ascending=False))
missing_nonzero = missing[missing.missing_count > 0].reset_index(names="column")
if not missing_nonzero.empty:
    px.bar(missing_nonzero, x="column", y="missing_pct",
           title="Missing values by column (%)").show()
else:
    print("No missing values detected.")

,missing_count,missing_pct
ADDRESSLINE2,2521,89.30
STATE,1486,52.64
TERRITORY,1074,38.04
POSTALCODE,76,2.69
ORDERNUMBER,0,0.00
CUSTOMERNAME,0,0.00
CONTACTFIRSTNAME,0,0.00
CONTACTLASTNAME,0,0.00
COUNTRY,0,0.00
CITY,0,0.00


## 8. Descriptive statistics

In [20]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
if numeric_cols:
    display(df[numeric_cols].describe().T)
else:
    print("No numeric columns.")
if categorical_cols:
    cat_stats = pd.DataFrame({
        "column": categorical_cols,
        "unique_values": [df[c].nunique(dropna=True) for c in categorical_cols],
        "most_common": [
            df[c].mode(dropna=True).iloc[0] if not df[c].mode(dropna=True).empty else None
            for c in categorical_cols
        ]
    })
    display(cat_stats)

,count,mean,std,min,25%,50%,75%,max
ORDERNUMBER,2823.0,10258.725115,92.085478,10100.00,10180.00,10262.0,10333.5,10425.0
QUANTITYORDERED,2823.0,35.092809,9.741443,6.00,27.00,35.0,43.0,97.0
PRICEEACH,2823.0,83.658544,20.174277,26.88,68.86,95.7,100.0,100.0
ORDERLINENUMBER,2823.0,6.466171,4.225841,1.00,3.00,6.0,9.0,18.0
SALES,2823.0,3553.889072,1841.865106,482.13,2203.43,3184.8,4508.0,14082.8
QTR_ID,2823.0,2.717676,1.203878,1.00,2.00,3.0,4.0,4.0
MONTH_ID,2823.0,7.092455,3.656633,1.00,4.00,8.0,11.0,12.0
YEAR_ID,2823.0,2003.815090,0.699670,2003.00,2003.00,2004.0,2004.0,2005.0
MSRP,2823.0,100.715551,40.187912,33.00,68.00,99.0,124.0,214.0


,column,unique_values,most_common
0,ORDERDATE,252,11/14/2003 0:00
1,STATUS,6,Shipped
2,PRODUCTLINE,7,Classic Cars
3,PRODUCTCODE,109,S18_3232
4,CUSTOMERNAME,92,Euro Shopping Channel
5,PHONE,91,(91) 555 94 44
6,ADDRESSLINE1,92,"C/ Moralzarzal, 86"
7,ADDRESSLINE2,9,Level 3
8,CITY,73,Madrid
9,STATE,16,CA


## 9. Numeric distributions

In [21]:
for col in numeric_cols[:30]:
    vals = df[col].dropna()
    if len(vals):
        px.histogram(x=vals, nbins=40, title=f"Distribution: {col}",
                     labels={"x": col, "count": "Frequency"}).show()

## 10. Categorical distributions

In [22]:
for col in categorical_cols[:20]:
    counts = df[col].astype("string").fillna("<missing>").value_counts().head(20)
    if len(counts):
        fig = px.bar(x=counts.index.astype(str), y=counts.values,
                     title=f"Top values: {col}",
                     labels={"x": col, "y": "Count"})
        fig.update_layout(xaxis_tickangle=-45)
        fig.show()

## 11. Correlation analysis

In [23]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()
    px.imshow(corr, text_auto=".2f", aspect="auto",
              title="Numeric correlation matrix",
              color_continuous_scale="RdBu_r", zmin=-1, zmax=1).show()
    display(corr)
else:
    print("Need at least two numeric columns.")

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,QTR_ID,MONTH_ID,YEAR_ID,MSRP
ORDERNUMBER,1.000000,0.065543,-0.002935,-0.055550,0.039919,-0.051383,-0.039723,0.904596,-0.010280
QUANTITYORDERED,0.065543,1.000000,0.005564,-0.018397,0.551426,-0.035323,-0.039048,0.069535,0.017881
PRICEEACH,-0.002935,0.005564,1.000000,-0.020965,0.657841,0.008712,0.005152,-0.005938,0.670625
ORDERLINENUMBER,-0.055550,-0.018397,-0.020965,1.000000,-0.058400,0.040716,0.034016,-0.057367,-0.021067
SALES,0.039919,0.551426,0.657841,-0.058400,1.000000,-0.006796,-0.009605,0.035647,0.635239
QTR_ID,-0.051383,-0.035323,0.008712,0.040716,-0.006796,1.000000,0.979300,-0.433052,0.010234
MONTH_ID,-0.039723,-0.039048,0.005152,0.034016,-0.009605,0.979300,1.000000,-0.430163,0.008170
YEAR_ID,0.904596,0.069535,-0.005938,-0.057367,0.035647,-0.433052,-0.430163,1.000000,-0.014310
MSRP,-0.010280,0.017881,0.670625,-0.021067,0.635239,0.010234,0.008170,-0.014310,1.000000


In [24]:
outlier_rows = []
for col in numeric_cols:
    s = df[col].dropna()
    if s.empty: continue
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    count = int(((s < low) | (s > high)).sum())
    outlier_rows.append({
        "column": col, "lower_bound": low, "upper_bound": high,
        "possible_outliers": count,
        "outlier_pct": round(count/len(s)*100, 2)
    })
outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)

,column,lower_bound,upper_bound,possible_outliers,outlier_pct
0,ORDERNUMBER,9949.750,10563.750,0,0.00
1,QUANTITYORDERED,3.000,67.000,8,0.28
2,PRICEEACH,22.150,146.710,0,0.00
3,ORDERLINENUMBER,-6.000,18.000,0,0.00
4,SALES,-1253.425,7964.855,81,2.87
5,QTR_ID,-1.000,7.000,0,0.00
6,MONTH_ID,-6.500,21.500,0,0.00
7,YEAR_ID,2001.500,2005.500,0,0.00
8,MSRP,-16.000,208.000,28,0.99


## 13. Date detection and time trends

In [25]:
date_candidates = []
for col in df.columns:
    if pd.api.types.is_datetime64_any_dtype(df[col]):
        date_candidates.append(col)
    elif any(k in str(col).lower() for k in ["date", "time", "timestamp"]):
        parsed = pd.to_datetime(df[col], errors="coerce")
        if parsed.notna().mean() >= .8:
            date_candidates.append(col)
date_candidates = list(dict.fromkeys(date_candidates))
print("Date candidates:", date_candidates)

DATE_COLUMN = date_candidates[0] if date_candidates else None
if DATE_COLUMN:
    temp = df.copy()
    temp[DATE_COLUMN] = pd.to_datetime(temp[DATE_COLUMN], errors="coerce")
    temp = temp.dropna(subset=[DATE_COLUMN]).sort_values(DATE_COLUMN)
    daily_records = temp.set_index(DATE_COLUMN).resample("D").size()
    px.line(x=daily_records.index, y=daily_records.values,
            title="Records over time",
            labels={"x": DATE_COLUMN, "y": "Record count"}).show()
else:
    print("No reliable date field detected; time-series exploration skipped.")

Date candidates: ['ORDERDATE']


## 14. Business-field suggestions


In [26]:
BUSINESS_TERMS = {
    "date": ["date", "order_date", "invoice_date", "transaction_date"],
    "revenue": ["revenue", "sales_amount", "total_sales", "sales", "turnover", "amount"],
    "quantity": ["quantity", "qty", "units_sold", "units", "volume"],
    "cost": ["cost", "purchase_cost", "cogs", "expense"],
    "profit": ["profit", "gross_profit", "net_profit", "margin"],
    "product": ["product", "item", "sku"],
    "customer": ["customer", "client", "buyer", "customer_id"],
    "category": ["category", "segment", "department"],
    "order_id": ["order_id", "invoice_id", "transaction_id", "bill_no"]
}
suggestions = []
for role, terms in BUSINESS_TERMS.items():
    matches = []
    for col in df.columns:
        normalized = re.sub(r"[^a-z0-9]+", "_", str(col).lower()).strip("_")
        if any(t == normalized or t in normalized for t in terms):
            matches.append(col)
    suggestions.append({"business_role": role, "suggested_columns": matches,
                        "requires_confirmation": True})
business_suggestions = pd.DataFrame(suggestions)
display(business_suggestions)

,business_role,suggested_columns,requires_confirmation
0,date,[ORDERDATE],True
1,revenue,[SALES],True
2,quantity,[QUANTITYORDERED],True
3,cost,[],True
4,profit,[],True
5,product,"[PRODUCTLINE, PRODUCTCODE]",True
6,customer,[CUSTOMERNAME],True
7,category,[],True
8,order_id,[],True


## 15. Confirm business columns (optional)
Fill these values only after reviewing the suggestions. Leave `None` if unavailable.

In [27]:
DATE_COL = DATE_COLUMN
REVENUE_COL = None       # e.g. "Revenue"
QUANTITY_COL = None      # e.g. "Quantity"
PRODUCT_COL = None       # e.g. "Product"
CATEGORY_COL = None      # e.g. "Category"
COST_COL = None          # e.g. "Cost"
PROFIT_COL = None        # e.g. "Profit"

def valid_col(name):
    return name if name and name in df.columns else None

metrics = {}
for role, col in {
    "revenue": REVENUE_COL, "quantity": QUANTITY_COL,
    "cost": COST_COL, "profit": PROFIT_COL
}.items():
    col = valid_col(col)
    if col:
        metrics[f"total_{role}"] = float(pd.to_numeric(df[col], errors="coerce").sum())
print("Confirmed metrics:", metrics)

Confirmed metrics: {}


## 16. Business charts from confirmed mappings

In [28]:
if valid_col(REVENUE_COL):
    chart_df = df.copy()
    chart_df[REVENUE_COL] = pd.to_numeric(chart_df[REVENUE_COL], errors="coerce")
    if valid_col(DATE_COL):
        chart_df[DATE_COL] = pd.to_datetime(chart_df[DATE_COL], errors="coerce")
        trend = (chart_df.dropna(subset=[DATE_COL])
                 .groupby(DATE_COL)[REVENUE_COL].sum().reset_index())
        px.line(trend, x=DATE_COL, y=REVENUE_COL,
                title="Revenue over time").show()
    if valid_col(PRODUCT_COL):
        top = (chart_df.groupby(PRODUCT_COL)[REVENUE_COL]
               .sum().nlargest(20).reset_index())
        px.bar(top, x=PRODUCT_COL, y=REVENUE_COL,
               title="Top products by revenue").show()
    if valid_col(CATEGORY_COL):
        by_cat = (chart_df.groupby(CATEGORY_COL)[REVENUE_COL]
                  .sum().reset_index())
        px.bar(by_cat, x=CATEGORY_COL, y=REVENUE_COL,
               title="Revenue by category").show()
else:
    print("No confirmed revenue field. Generic EDA charts are available above.")

No confirmed revenue field. Generic EDA charts are available above.


## 17. Cleaning recommendations — no automatic destructive changes

In [29]:
recommendations = []
dups = int(df.duplicated().sum())
if dups:
    recommendations.append({
        "issue": "Exact duplicate rows", "count": dups,
        "recommendation": "Review whether these are accidental duplicates before removal."
    })
for col in df.columns:
    miss = int(df[col].isna().sum())
    if miss:
        recommendations.append({
            "issue": f"Missing values: {col}", "count": miss,
            "recommendation": "Inspect missingness and column meaning before imputing or excluding."
        })
    if df[col].nunique(dropna=True) <= 1:
        recommendations.append({
            "issue": f"Constant/empty column: {col}",
            "count": int(df[col].notna().sum()),
            "recommendation": "Consider removal only after confirming it has no business/audit value."
        })
cleaning_recommendations = pd.DataFrame(recommendations)
display(cleaning_recommendations if len(cleaning_recommendations) else
        pd.DataFrame({"status": ["No basic cleaning recommendations detected"]}))

,issue,count,recommendation
0,Missing values: ADDRESSLINE2,2521,Inspect missingness and column meaning before imputing or excluding.
1,Missing values: STATE,1486,Inspect missingness and column meaning before imputing or excluding.
2,Missing values: POSTALCODE,76,Inspect missingness and column meaning before imputing or excluding.
3,Missing values: TERRITORY,1074,Inspect missingness and column meaning before imputing or excluding.


## 18. Create a review copy (optional)
This does not overwrite the source. Exact deduplication is off by default.

In [30]:
APPLY_EXACT_DEDUPLICATION = False
df_review = df.copy(deep=True)
if APPLY_EXACT_DEDUPLICATION:
    before = len(df_review)
    df_review = df_review.drop_duplicates().copy()
    print("Removed exact duplicates:", before - len(df_review))
else:
    print("No changes applied.")
review_path = OUTPUT_DIR / "dataset_review_copy.csv"
df_review.to_csv(review_path, index=False)
print("Review copy saved:", review_path.resolve())

No changes applied.
Review copy saved: D:\Programing\Programing_folder\Program Files\projectfile\vayapar_sangisarathi\notebooks\vss_ai_outputs\dataset_review_copy.csv


## 19. ML-readiness checks


In [31]:
numeric = df.select_dtypes(include=np.number).columns.tolist()
categorical = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
readiness = pd.DataFrame([
    ["Has records", len(df) > 0, f"{len(df):,} rows"],
    ["Has columns", df.shape[1] > 0, f"{df.shape[1]} columns"],
    ["Numeric features exist", len(numeric) > 0, f"{len(numeric)} numeric columns"],
    ["Categorical features exist", len(categorical) > 0, f"{len(categorical)} categorical columns"],
    ["Average missingness < 30%", df.isna().mean().mean() < .30,
     f"{df.isna().mean().mean()*100:.1f}% average cell missingness"],
    ["At least 100 rows", len(df) >= 100, f"{len(df):,} rows"]
], columns=["check", "passed", "details"])
display(readiness)
print("Before ML: choose a target, prevent leakage, define prediction time, and use an appropriate train/test split.")

,check,passed,details
0,Has records,True,"2,823 rows"
1,Has columns,True,25 columns
2,Numeric features exist,True,9 numeric columns
3,Categorical features exist,True,16 categorical columns
4,Average missingness < 30%,True,7.3% average cell missingness
5,At least 100 rows,True,"2,823 rows"


Before ML: choose a target, prevent leakage, define prediction time, and use an appropriate train/test split.


## 20. Export analysis outputs

In [32]:
column_profile.to_csv(OUTPUT_DIR / "column_profile.csv", index=False)
quality_report.to_csv(OUTPUT_DIR / "data_quality_report.csv", index=False)
missing.to_csv(OUTPUT_DIR / "missing_value_summary.csv")
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False)
business_suggestions.to_csv(OUTPUT_DIR / "business_field_suggestions.csv", index=False)
readiness.to_csv(OUTPUT_DIR / "ml_readiness_report.csv", index=False)
cleaning_recommendations.to_csv(OUTPUT_DIR / "cleaning_recommendations.csv", index=False)

summary = {
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "duplicate_rows": int(df.duplicated().sum()),
    "numeric_columns": numeric_cols,
    "categorical_columns": categorical_cols,
    "date_candidates": date_candidates,
    "confirmed_metrics": metrics,
    "ml_readiness": readiness.to_dict(orient="records")
}
with open(OUTPUT_DIR / "dataset_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

print("Saved outputs to:", OUTPUT_DIR.resolve())
print([p.name for p in OUTPUT_DIR.iterdir() if p.is_file()])

Saved outputs to: D:\Programing\Programing_folder\Program Files\projectfile\vayapar_sangisarathi\notebooks\vss_ai_outputs
['business_field_suggestions.csv', 'cleaning_recommendations.csv', 'column_profile.csv', 'dataset_review_copy.csv', 'dataset_summary.json', 'data_quality_report.csv', 'missing_value_summary.csv', 'ml_readiness_report.csv', 'outlier_report.csv']
